# NB01: Data cleaning with full scientific filters

ExPetDB opx experimental data, cleaned to equilibrium-filtered, deduplicated, stability-constrained training set.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

In [2]:
# Set up logger
log_path = LOGS / 'cleaning_log.txt'
log_lines = []
def log(msg):
    print(msg)
    log_lines.append(str(msg))

log(f'NB01 start: {pd.Timestamp.now()}')
log(f'ExPetDB source: {EXPETDB}')

NB01 start: 2026-04-14 02:45:20.989137
ExPetDB source: C:\Users\NQTa\Documents\MLCourse\Final Project\data\raw\ExPetDB_download_ExPetDB-2025-07-21.xlsx


In [3]:
# STEP 1: Load sheets and merge
exp = pd.read_excel(EXPETDB, sheet_name='Experiment')
opx = pd.read_excel(EXPETDB, sheet_name='Orthopyroxene')
log(f'\nSTEP 1: Load sheets')
log(f'  Experiment: {len(exp)} rows')
log(f'  Orthopyroxene: {len(opx)} rows')

# Keep only opx rows with T and P from Experiment sheet
exp_small = exp[['Experiment', 'Citation', 'DOI', 'T (C)', 'P (GPa)']].copy()
exp_small = exp_small.rename(columns={'T (C)': 'T_C', 'P (GPa)': 'P_GPa'})
exp_small['T_C'] = pd.to_numeric(exp_small['T_C'], errors='coerce')
exp_small['P_GPa'] = pd.to_numeric(exp_small['P_GPa'], errors='coerce')
exp_small['P_kbar'] = exp_small['P_GPa'] * 10.0  # 1 GPa = 10 kbar
exp_small = exp_small.dropna(subset=['T_C', 'P_kbar'])

# Merge opx with experiment conditions
df = opx.merge(exp_small, on='Experiment', how='inner', suffixes=('', '_exp'))
if 'Citation_exp' in df.columns:
    df['Citation'] = df['Citation'].fillna(df['Citation_exp'])
    df = df.drop(columns=['Citation_exp'])
log(f'  Merged opx+experiment: {len(df)} rows')


STEP 1: Load sheets
  Experiment: 16082 rows
  Orthopyroxene: 1796 rows
  Merged opx+experiment: 7466 rows


In [4]:
# STEP 2: Extract oxides and handle iron
log(f'\nSTEP 2: Extract oxides')
oxide_cols_raw = ['SiO2', 'TiO2', 'Al2O3', 'Cr2O3', 'Fe2O3', 'FeO', 'MnO', 'MgO', 'CaO', 'Na2O', 'K2O']
for ox in oxide_cols_raw:
    col = f'{ox} value'
    if col in df.columns:
        df[ox] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    else:
        df[ox] = 0.0

# FeO_total = FeO + 0.8998 * Fe2O3
df['FeO_total'] = df['FeO'].fillna(0) + 0.8998 * df['Fe2O3'].fillna(0)
log(f'  Iron combined: FeO_total = FeO + 0.8998*Fe2O3')
log(f'  Mean FeO_total: {df["FeO_total"].mean():.2f} wt%')


STEP 2: Extract oxides
  Iron combined: FeO_total = FeO + 0.8998*Fe2O3
  Mean FeO_total: 10.26 wt%


In [5]:
# STEP 2b: Fe3+ protocol (fixed ratio)
log(f'\nSTEP 2b: Fe3+ protocol')
df['Fe3_FeT'] = FE3_FET_RATIO
df['FeO_calc'] = df['FeO_total'] * (1 - FE3_FET_RATIO)
# Fe2O3 = FeO_total * Fe3/FeT * (MW_Fe2O3 / (2*MW_FeO))
df['Fe2O3_calc'] = df['FeO_total'] * FE3_FET_RATIO * (159.69 / 71.844 / 2)
log(f'  Fe3+/FeT = {FE3_FET_RATIO} applied to {len(df)} rows')


STEP 2b: Fe3+ protocol
  Fe3+/FeT = 0.15 applied to 7466 rows


In [6]:
# STEP 2c: is_vbd flag
log(f'\nSTEP 2c: Volume-by-difference (VBD) flag')
# ExPetDB Orthopyroxene sheet does not carry H2O method per-spot; flag False for all
# (The is_vbd distinction applies at the liquid/bulk level.)
df['is_vbd'] = False
log(f'  Opx spots: is_vbd = False (VBD is a liquid-level concept in ExPetDB)')


STEP 2c: Volume-by-difference (VBD) flag
  Opx spots: is_vbd = False (VBD is a liquid-level concept in ExPetDB)


In [7]:
# STEP 3: Oxide total filter
log(f'\nSTEP 3: Oxide total filter ({OXIDE_TOTAL_MIN}-{OXIDE_TOTAL_MAX} wt%)')
df['oxide_total'] = df[oxide_cols_raw].sum(axis=1)
n_before = len(df)
df = df[df['oxide_total'].between(OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX)].copy()
log(f'  Dropped {n_before - len(df)} rows outside {OXIDE_TOTAL_MIN}-{OXIDE_TOTAL_MAX}')
log(f'  Remaining: {len(df)}')


STEP 3: Oxide total filter (95.0-102.0 wt%)
  Dropped 177 rows outside 95.0-102.0
  Remaining: 7289


In [8]:
# STEP 4: Cation recalculation on 6-oxygen basis
log(f'\nSTEP 4: Cation recalculation (6-oxygen basis)')

# Molecular weights and (cations, oxygens) per oxide formula unit
ox_data = {
    'SiO2':  (60.084,   1, 2),
    'TiO2':  (79.866,   1, 2),
    'Al2O3': (101.961,  2, 3),
    'Cr2O3': (151.99,   2, 3),
    'FeO':   (71.844,   1, 1),   # using FeO_total as the iron oxide input
    'MnO':   (70.937,   1, 1),
    'MgO':   (40.304,   1, 1),
    'CaO':   (56.077,   1, 1),
    'Na2O':  (61.979,   2, 1),
    'K2O':   (94.195,   2, 1),
}

# Use FeO_total as the iron input
iron_col = df['FeO_total']

def cation_recalc(row):
    mol_cat = {}
    mol_ox = 0.0
    for ox, (mw, n_cat, n_ox) in ox_data.items():
        wt = row['FeO_total'] if ox == 'FeO' else row[ox]
        wt = float(wt) if pd.notna(wt) else 0.0
        moles_ox_unit = wt / mw
        mol_cat[ox] = moles_ox_unit * n_cat
        mol_ox += moles_ox_unit * n_ox
    if mol_ox <= 0:
        return {k + '_cat': 0.0 for k in ox_data} | {'cation_sum': 0.0}
    factor = 6.0 / mol_ox
    out = {ox + '_cat': mol_cat[ox] * factor for ox in ox_data}
    out['cation_sum'] = sum(out.values())
    return out

cat_df = pd.DataFrame([cation_recalc(r) for _, r in df.iterrows()], index=df.index)
df = pd.concat([df, cat_df], axis=1)
log(f'  Cation recalc done. Mean cation sum: {df["cation_sum"].mean():.3f}')


STEP 4: Cation recalculation (6-oxygen basis)


  Cation recalc done. Mean cation sum: 4.000


In [9]:
# STEP 5: Core oxide requirement + cation sum filter
log(f'\nSTEP 5: Core oxide requirement + cation sum filter')
core_required = ['SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO']
n_before = len(df)
# Core oxides must all be > 0 (i.e., measured)
core_ok = (df[core_required] > 0).all(axis=1)
df = df[core_ok].copy()
log(f'  Dropped {n_before - len(df)} rows missing core oxides')

n_before = len(df)
df = df[df['cation_sum'].between(CATION_SUM_MIN, CATION_SUM_MAX)].copy()
log(f'  Dropped {n_before - len(df)} rows outside cation sum {CATION_SUM_MIN}-{CATION_SUM_MAX}')
log(f'  Remaining: {len(df)}')


STEP 5: Core oxide requirement + cation sum filter
  Dropped 2737 rows missing core oxides
  Dropped 26 rows outside cation sum 3.95-4.05
  Remaining: 4526


In [10]:
# STEP 6: Pigeonite filter (Wo <= 5 mol%)
log(f'\nSTEP 6: Pigeonite filter (Wo <= {WO_MAX_MOL_PCT} mol%)')
# Wo, En, Fs on a cation basis, as mol%
mg = df['MgO_cat']
fe = df['FeO_cat']
ca = df['CaO_cat']
denom = (mg + fe + ca).replace(0, np.nan)
df['En_frac'] = 100 * mg / denom
df['Fs_frac'] = 100 * fe / denom
df['Wo_frac'] = 100 * ca / denom
df['Wo'] = df['Wo_frac']  # alias used by assertions

n_before = len(df)
df = df[df['Wo'] <= WO_MAX_MOL_PCT].copy()
log(f'  Dropped {n_before - len(df)} pigeonite/aug (Wo > {WO_MAX_MOL_PCT})')
log(f'  Remaining: {len(df)}')


STEP 6: Pigeonite filter (Wo <= 5.0 mol%)
  Dropped 374 pigeonite/aug (Wo > 5.0)
  Remaining: 4152


In [11]:
# STEP 6b: Pressure ceiling
log(f'\nSTEP 6b: Pressure ceiling filter (P <= {P_CEILING_KBAR} kbar)')
n_before = len(df)
df = df[df['P_kbar'] <= P_CEILING_KBAR].copy()
log(f'  Dropped {n_before - len(df)} rows with P > {P_CEILING_KBAR} kbar (opx unstable)')
log(f'  Remaining: {len(df)}')


STEP 6b: Pressure ceiling filter (P <= 100.0 kbar)
  Dropped 71 rows with P > 100.0 kbar (opx unstable)
  Remaining: 4081


In [12]:
# STEP 6c: KD(Fe-Mg) opx-liq equilibrium filter (requires liquid merge)
log(f'\nSTEP 6c: KD(Fe-Mg) opx-liq equilibrium filter ({KD_FEMG_MIN}-{KD_FEMG_MAX})')

liq_raw = pd.read_excel(EXPETDB, sheet_name='Liquid')
liq_cols_raw = [f'{o} value' for o in LIQ_OXIDES]
liq = liq_raw[['Experiment'] + liq_cols_raw + ['H2O value', 'H2O method']].copy()
liq.columns = ['Experiment'] + [f'liq_{o}' for o in LIQ_OXIDES] + ['H2O_Liq', 'H2O_Liq_Method']
for c in [f'liq_{o}' for o in LIQ_OXIDES]:
    liq[c] = pd.to_numeric(liq[c], errors='coerce')
liq['H2O_Liq'] = pd.to_numeric(liq['H2O_Liq'], errors='coerce')

# Dedup liquid: keep row with oxide total closest to 100
liq_cols = [f'liq_{o}' for o in LIQ_OXIDES]
liq['_total'] = liq[liq_cols].sum(axis=1, min_count=5)
liq['_dist_100'] = (liq['_total'] - 100).abs()
liq = liq.sort_values(['Experiment', '_dist_100']).drop_duplicates(subset='Experiment', keep='first')
liq = liq.drop(columns=['_total', '_dist_100'])

df = df.merge(liq, on='Experiment', how='left')

mw_MgO, mw_FeO = 40.304, 71.844
has_liq = df['liq_MgO'].notna() & df['liq_FeO'].notna() & (df['liq_MgO'] > 0) & (df['liq_FeO'] > 0)
df['FeMg_opx'] = np.nan
df['FeMg_liq'] = np.nan
df['KD_FeMg'] = np.nan
df.loc[has_liq, 'FeMg_opx'] = (df.loc[has_liq, 'FeO_total']/mw_FeO) / (df.loc[has_liq, 'MgO']/mw_MgO)
df.loc[has_liq, 'FeMg_liq'] = (df.loc[has_liq, 'liq_FeO']/mw_FeO) / (df.loc[has_liq, 'liq_MgO']/mw_MgO)
df.loc[has_liq, 'KD_FeMg'] = df.loc[has_liq, 'FeMg_opx'] / df.loc[has_liq, 'FeMg_liq']

df['opx_liq_pair'] = has_liq & (df['KD_FeMg'] >= KD_FEMG_MIN) & (df['KD_FeMg'] <= KD_FEMG_MAX)

# is_vbd from liquid H2O method
df['is_vbd'] = df['H2O_Liq_Method'].astype(str).str.contains('VBD|diff|mass_balance|calculated', case=False, na=False, regex=True)

log(f'  Rows with liquid data: {int(has_liq.sum())}')
log(f'  Rows passing KD filter (opx_liq_pair=True): {int(df["opx_liq_pair"].sum())}')
log(f'  is_vbd (liquid H2O by difference): {int(df["is_vbd"].sum())}')
log(f'  Opx-only rows (total): {len(df)}')


STEP 6c: KD(Fe-Mg) opx-liq equilibrium filter (0.23-0.35)


  Rows with liquid data: 3745
  Rows passing KD filter (opx_liq_pair=True): 1036
  is_vbd (liquid H2O by difference): 64
  Opx-only rows (total): 4081


In [13]:
# STEP 6d: Experiment-level deduplication
log(f'\nSTEP 6d: Experiment-level deduplication')
n_before = len(df)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
string_cols = df.select_dtypes(include=['object']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

agg_dict = {c: 'mean' for c in numeric_cols}
agg_dict.update({c: 'first' for c in string_cols})
agg_dict.update({c: 'any' for c in bool_cols})
agg_dict.pop('Experiment', None)

df = df.groupby('Experiment', as_index=False).agg(agg_dict)
log(f'  Before dedup: {n_before} spot-level rows')
log(f'  After dedup: {len(df)} experiment-level rows')
log(f'  Collapsed {n_before - len(df)} duplicate spots')


STEP 6d: Experiment-level deduplication
  Before dedup: 4081 spot-level rows
  After dedup: 1035 experiment-level rows
  Collapsed 3046 duplicate spots


In [14]:
# STEP 7: Feature engineering (additional geochem features)
log(f'\nSTEP 7: Feature engineering')

# Mg# (cation basis) using FeO_total
df['Mg_num'] = 100 * df['MgO_cat'] / (df['MgO_cat'] + df['FeO_cat']).replace(0, np.nan)

# Al site distribution: 2 tetrahedral sites per formula unit
df['Al_IV'] = (2.0 - df['SiO2_cat']).clip(lower=0)
df['Al_VI'] = (df['Al2O3_cat'] - df['Al_IV']).clip(lower=0)

# MgTs component proxy: limited by Al_IV (pure Mg-Tschermak is MgAl(AlSi)O6)
df['MgTs'] = np.minimum(df['Al_IV'], df['Al_VI']).clip(lower=0)

# Liquid Mg# if liquid present
has_liq = df['liq_MgO'].notna() & df['liq_FeO'].notna() & (df['liq_MgO'] > 0) & (df['liq_FeO'] > 0)
df['liq_Mg_num'] = np.nan
df.loc[has_liq, 'liq_Mg_num'] = 100 * (df.loc[has_liq, 'liq_MgO']/40.304) / (
    df.loc[has_liq, 'liq_MgO']/40.304 + df.loc[has_liq, 'liq_FeO']/71.844)

log(f'  Features added: Mg_num, Al_IV, Al_VI, MgTs, liq_Mg_num')


STEP 7: Feature engineering
  Features added: Mg_num, Al_IV, Al_VI, MgTs, liq_Mg_num


In [15]:
# STEP 8: Save datasets
log(f'\nSTEP 8: Save datasets')

# Core: all 5 core oxides present (which is already enforced in Step 5)
df_core = df.copy()

# Full: requires all 9 OPX_FULL_OXIDES present
full_mask = (df[OPX_FULL_OXIDES] > 0).all(axis=1)
df_full = df[full_mask].copy()

df_core.to_csv(DATA_PROC / 'opx_clean_core.csv', index=False)
df_core.to_parquet(DATA_PROC / 'opx_clean_core.parquet')
df_full.to_csv(DATA_PROC / 'opx_clean_full.csv', index=False)
df_full.to_parquet(DATA_PROC / 'opx_clean_full.parquet')

log(f'  Core: {len(df_core)} experiments, {df_core["Citation"].nunique()} studies')
log(f'  Core opx_liq_pair=True: {int(df_core["opx_liq_pair"].sum())}')
log(f'  Full: {len(df_full)} experiments, {df_full["Citation"].nunique()} studies')
log(f'  Full opx_liq_pair=True: {int(df_full["opx_liq_pair"].sum())}')

with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))
log(f'\nLog written to {log_path}')


STEP 8: Save datasets


  Core: 1035 experiments, 123 studies
  Core opx_liq_pair=True: 600
  Full: 494 experiments, 70 studies
  Full opx_liq_pair=True: 321

Log written to C:\Users\NQTa\Documents\MLCourse\Final Project\logs\cleaning_log.txt


In [16]:
# Verification
assert df_core['P_kbar'].max() <= P_CEILING_KBAR, f"P ceiling filter failed: max={df_core['P_kbar'].max()}"
assert df_core['Wo'].max() <= WO_MAX_MOL_PCT, f"Pigeonite filter failed: max={df_core['Wo'].max()}"
assert df_core['cation_sum'].between(CATION_SUM_MIN, CATION_SUM_MAX).all(), "Cation sum filter failed"
print(f"Core: {len(df_core)} experiments, {df_core['Citation'].nunique()} studies")
print(f"  opx_liq_pair = True: {int(df_core['opx_liq_pair'].sum())}")
print(f"Full: {len(df_full)} experiments, {df_full['Citation'].nunique()} studies")
print("\n=== PHASE 1 COMPLETE ===")

Core: 1035 experiments, 123 studies
  opx_liq_pair = True: 600
Full: 494 experiments, 70 studies

=== PHASE 1 COMPLETE ===
